# GC 1-Minute – Barrier Strategy with Gradient Boosting – Indicator Period Test

Same as `train_barrier_GC.ipynb`, but with a wider set of RSI / TWAP / VWAP periods, permutation importance for the
direction model and an automatic comparison of the direction AUC (validation) with the previous run.
Results go to `models_barrier_indicators/` and `..._barrier_indicators_signals.csv`, so the earlier run is not overwritten.

Instead of forecasting the price, this notebook learns the trading rule directly. For every 1-minute candle and every setup:

> **Long:** is the price `+target` ticks above the entry (close of the candle) **before** it is `stop` ticks below,
> within `horizon` minutes? – **Short:** the same mirrored.

A target reached in minute 13 of a 15-minute window counts as a win. Two classifiers per setup (long, short) estimate these
probabilities; a trade is taken only when a probability is above a threshold that is chosen on the **validation period**
and then applied **once** to the separate test CSV.

**Inputs:** only the CSV columns Open, Close, High, Low, Ask, Bid, Delta, Delta/Volume, Session Delta, Session Delta %,
Max Delta, Min Delta, Delta Change, Volume – plus the time of day / weekday from the timestamp (`USE_TIME_FEATURES`).
All other CSV columns are ignored. The timestamp and all original columns stay in the output CSV.

**Time split without overlap:** training data = everything in the training CSV **before** the first candle of the test CSV.
Of that, the last `VALIDATION_FRACTION` is validation (threshold selection); the test CSV is only used at the very end.

**Comparison:** on the test data the model is compared with *random direction* and *always long* using the same entry
times and the same exit rules. The model is only useful if it is clearly better than both **after costs**.

## 0. Configuration

In [1]:
from pathlib import Path

SYMBOL = "GC"
TRAIN_CSV_PATH = Path(r"E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv")
TEST_CSV_PATH = Path(r"E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv")
TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH.with_name(TEST_CSV_PATH.stem + "_barrier_indicators_signals.csv")
MODEL_DIR = Path("models_barrier_indicators")   # separate folder -> earlier runs stay untouched
LOG_PATH = MODEL_DIR / "barrier_log.csv"

# Model inputs: ONLY these CSV columns (+ time features from "Time UTC" if USE_TIME_FEATURES)
INPUT_COLUMNS = [
    "Open", "Close", "High", "Low", "Ask", "Bid", "Delta", "Delta/Volume",
    "Session Delta", "Session Delta %", "Max Delta", "Min Delta", "Delta Change", "Volume",
]
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]   # exported as text "59.15%" -> 0.5915
USE_TIME_FEATURES = True                                 # hour of day + weekday from the timestamp

# Additional indicators (calculated only from the input columns above)
RSI_PERIODS = [3, 5, 7, 9, 14, 21, 30, 50, 100]   # RSI (Wilder) on 1-minute closes
TWAP_WINDOWS = [5, 15, 30, 60, 120, 240]          # rolling TWAP in minutes (+ session TWAP, reset after every pause)
VWAP_WINDOWS = [5, 15, 30, 60, 120, 240]          # rolling VWAP in minutes (+ session VWAP, reset after every pause)
# Direction AUC (validation) of the previous run with RSI 7/14/21/50 and TWAP/VWAP 15/60 – used as reference in 6.2.
# New periods are only worth keeping if the validation AUC rises by at least MIN_AUC_GAIN.
REFERENCE_DIRECTION_AUC = {"15min": 0.5228, "60min": 0.5152}
MIN_AUC_GAIN = 0.005

# Trade setups (horizon in minutes). Two kinds of barriers:
#   mode "vol":   target/stop = sigma x expected move over the horizon
#                 (std of 1-minute close changes over the last VOL_LOOKBACK candles x sqrt(horizon)), in ticks
#                 -> a trade means the same in quiet and in wild markets
#   mode "fixed": target_ticks / stop_ticks as fixed numbers, e.g.
#                 {"name": "15min_fix", "horizon": 15, "mode": "fixed", "target_ticks": 30, "stop_ticks": 30}
SETUPS = [
    {"name": "15min", "horizon": 15, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
    {"name": "60min", "horizon": 60, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
]
VOL_LOOKBACK = 60               # candles for the volatility estimate (only past data)
MIN_BARRIER_TICKS = 5           # volatility barriers are never smaller than this

# Futures contract and execution
TICK_SIZE = 0.10                # GC / MGC
TICK_VALUE_USD = 10.0           # GC: 10 USD per tick and contract (MGC: 1 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1     # target counts only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 5             # no trade if the window contains a longer pause (session break, weekend)
WARMUP_CANDLES = 240            # first candles of a file are not traded (longest rolling input: 240 minutes)

# Data split and training
VALIDATION_FRACTION = 0.2       # last 20 % of the training data (before the test period) -> threshold selection
TRAIN_ROW_STRIDE = 1            # 1 = every candle; e.g. 3 = every 3rd candle (faster, slightly less data)
GBM_PARAMS = dict(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, random_state=42,
)

# Trading rule: trade if max(P_long, P_short) >= threshold; the threshold is chosen on validation (best t-value)
PROB_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(21)]   # 0.30 ... 0.80
MIN_VALIDATION_TRADES = 100     # thresholds with fewer validation trades are not eligible
BASELINE_RANDOM_RUNS = 5
COMPUTE_FEATURE_IMPORTANCE = True   # permutation importance on a validation sample (a few minutes)

# Direction strategy (section 8.2): direction from the direction model, timing from the win models.
# All combinations are evaluated on validation; the one with the best t-value is applied ONCE to the test data.
DIRECTION_QUANTILES = [0.05, 0.10, 0.20]   # trade the lowest / highest X % of P(up first) -> short / long
TIMING_QUANTILES = [0.0, 0.5, 0.75]        # also require P(win) of the traded side >= this validation quantile (0 = off)
DIRECTION_SIDES = ["both", "short", "long"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
for p in (TRAIN_CSV_PATH, TEST_CSV_PATH):
    if not p.is_file():
        raise FileNotFoundError(p)
print(f"Train CSV: {TRAIN_CSV_PATH}\nTest CSV:  {TEST_CSV_PATH}")
for s in SETUPS:
    if s.get("mode", "fixed") == "vol":
        print(f"Setup {s['name']}: target {s['target_sigma']} σ / stop {s['stop_sigma']} σ of the expected "
              f"{s['horizon']}-minute move (ticks are shown after the labels are computed)")
    else:
        be = (s["stop_ticks"] + COST_PER_TRADE_TICKS) / (s["target_ticks"] + s["stop_ticks"])
        print(f"Setup {s['name']}: +{s['target_ticks']} / -{s['stop_ticks']} ticks within {s['horizon']} min "
              f"-> break-even win rate ≈ {100 * be:.1f} % (ignoring time exits)")

Train CSV: E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv
Test CSV:  E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv
Setup 15min: target 1.0 σ / stop 1.0 σ of the expected 15-minute move (ticks are shown after the labels are computed)
Setup 60min: target 1.0 σ / stop 1.0 σ of the expected 60-minute move (ticks are shown after the labels are computed)


## 1. Imports

In [2]:
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 2. Load CSV

In [3]:
def load_csv(csv_path: Path) -> pd.DataFrame:
    """Load an ATAS 1-minute export in Bar order; only the input columns, the timestamp and Bar are kept."""
    raw = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
    missing = [c for c in ["Bar", "Time UTC"] + INPUT_COLUMNS if c not in raw.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")

    for col in PERCENT_COLUMNS:
        text = (raw[col].astype(str).str.strip().str.replace("%", "", regex=False)
                .str.replace("'", "", regex=False).str.replace(" ", "", regex=False))
        text = text.where(text.str.contains(".", regex=False), text.str.replace(",", ".", regex=False))
        values = pd.to_numeric(text, errors="coerce")
        n_bad = int((values.isna() & (text != "") & (text.str.lower() != "nan")).sum())
        if n_bad:
            print(f"⚠ {col}: {n_bad:,} values not parseable -> 0")
        raw[col] = (values / 100).fillna(0.0)

    df = raw[["Bar", "Time UTC"] + INPUT_COLUMNS].copy()
    for col in INPUT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    df["timestamp"] = pd.to_datetime(df["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True)
    if not df["Bar"].is_monotonic_increasing:
        df = df.sort_values("Bar", kind="stable")
    df = df.drop(columns=["Time UTC"]).reset_index(drop=True)
    if df[["Open", "High", "Low", "Close"]].isna().any().any():
        raise ValueError(f"{csv_path.name}: missing OHLC values")
    print(f"{csv_path.name}: {len(df):,} candles, {df['timestamp'].min()} → {df['timestamp'].max()}")
    return df


df_all = load_csv(TRAIN_CSV_PATH)
df_test = load_csv(TEST_CSV_PATH)

# Training data = everything before the first candle of the test CSV (no overlap)
test_start = df_test["timestamp"].min()
df_train = df_all[df_all["timestamp"] < test_start].reset_index(drop=True)
del df_all
print(f"Training data used: {len(df_train):,} candles until {df_train['timestamp'].max()} (test starts {test_start})")

Candelexport_GC_M1_25_11_2018_20_10_2026.csv: 2,796,096 candles, 2018-11-25 23:00:00+00:00 → 2026-10-20 20:59:00+00:00
Test_GC_M1_20_10_2025_20_10_2026.csv: 354,649 candles, 2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00
Training data used: 2,441,447 candles until 2025-10-19 23:59:00+00:00 (test starts 2025-10-20 00:00:00+00:00)


## 3. Inputs (features)

Only backward-looking values of the input columns: candle shape, price changes, volatility and position in the recent range
(all in ticks, independent of the price level), plus order flow (Ask/Bid, Delta, Session Delta, Max/Min Delta, Volume)
as raw values, ratios and rolling sums. Gradient boosting handles missing values itself.

**Indicators:** RSI for every period in `RSI_PERIODS` (level + change over 5 minutes), **TWAP** and **VWAP** – each as
session value (reset at every session start / after a pause) and as rolling value over `TWAP_WINDOWS` / `VWAP_WINDOWS`.
TWAP/VWAP enter as **distance of the close in ticks** (and relative to the recent candle range), so the model does not
see absolute price levels. Typical price = (High + Low + Close) / 3.

In [4]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    """Relative Strength Index with Wilder smoothing (0..100)."""
    diff = close.diff()
    avg_gain = diff.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = (-diff.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    value[(avg_loss == 0) & (avg_gain > 0)] = 100.0
    value[(avg_loss == 0) & (avg_gain == 0)] = 50.0
    return value


def session_ids(ts: pd.Series) -> pd.Series:
    """New session after every pause > MAX_GAP_MINUTES (daily break, weekend)."""
    return (ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES).cumsum()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    vol, ask, bid, delta = df["Volume"], df["Ask"], df["Bid"], df["Delta"]
    t = TICK_SIZE
    f = {}

    # Candle shape and price changes (ticks)
    f["body_ticks"] = (c - o) / t
    f["range_ticks"] = (h - l) / t
    f["upper_wick_ticks"] = (h - np.maximum(o, c)) / t
    f["lower_wick_ticks"] = (np.minimum(o, c) - l) / t
    ret1 = (c - c.shift(1)) / t
    for k in [1, 3, 5, 10, 15, 30, 60]:
        f[f"ret_{k}_ticks"] = (c - c.shift(k)) / t
    for k in [5, 15, 60]:
        hh, ll = h.rolling(k).max(), l.rolling(k).min()
        f[f"range_mean_{k}"] = f["range_ticks"].rolling(k).mean()
        f[f"ret_std_{k}"] = ret1.rolling(k).std()
        f[f"pos_in_range_{k}"] = (c - ll) / (hh - ll).replace(0, np.nan)
        f[f"dist_high_{k}_ticks"] = (hh - c) / t
        f[f"dist_low_{k}_ticks"] = (c - ll) / t

    # Order flow
    f["volume"] = vol
    f["log_volume"] = np.log1p(vol.clip(lower=0))
    f["ask"] = ask
    f["bid"] = bid
    f["ask_bid_imbalance"] = (ask - bid) / (ask + bid).replace(0, np.nan)
    f["delta"] = delta
    f["delta_per_volume"] = df["Delta/Volume"]
    f["session_delta"] = df["Session Delta"]
    f["session_delta_pct"] = df["Session Delta %"]
    f["session_delta_change_15"] = df["Session Delta"] - df["Session Delta"].shift(15)
    f["max_delta"] = df["Max Delta"]
    f["min_delta"] = df["Min Delta"]
    f["delta_change"] = df["Delta Change"]
    f["max_delta_rel"] = df["Max Delta"] / vol.replace(0, np.nan)
    f["min_delta_rel"] = df["Min Delta"] / vol.replace(0, np.nan)
    for k in [1, 2, 3, 5]:
        f[f"delta_lag_{k}"] = delta.shift(k)
        f[f"delta_per_volume_lag_{k}"] = df["Delta/Volume"].shift(k)
    for k in [5, 15, 60]:
        f[f"delta_sum_{k}"] = delta.rolling(k).sum()
        f[f"delta_ratio_{k}"] = delta.rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
        f[f"volume_rel_{k}"] = vol / vol.rolling(k).mean().replace(0, np.nan)
        f[f"ask_bid_imbalance_{k}"] = ((ask - bid).rolling(k).sum()
                                       / (ask + bid).rolling(k).sum().replace(0, np.nan))

    # RSI
    for p in RSI_PERIODS:
        r = rsi(c, p)
        f[f"rsi_{p}"] = r
        f[f"rsi_{p}_change_5"] = r - r.shift(5)

    # TWAP / VWAP (distance of the close in ticks and relative to the recent candle range)
    tp = (h + l + c) / 3
    session = session_ids(df["timestamp"])
    range_60 = ((h - l) / t).rolling(60, min_periods=5).mean().replace(0, np.nan)
    twap_session = tp.groupby(session).cumsum() / (tp.groupby(session).cumcount() + 1)
    vol_cum = vol.groupby(session).cumsum()
    vwap_session = ((tp * vol).groupby(session).cumsum() / vol_cum.replace(0, np.nan)).groupby(session).ffill()
    averages = {"twap_session": twap_session, "vwap_session": vwap_session}
    for k in TWAP_WINDOWS:
        averages[f"twap_{k}"] = tp.rolling(k).mean()
    for k in VWAP_WINDOWS:
        averages[f"vwap_{k}"] = (tp * vol).rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
    for name, avg in averages.items():
        f[f"dist_{name}_ticks"] = (c - avg) / t
        f[f"dist_{name}_rel"] = (c - avg) / t / range_60
    f["vwap_minus_twap_session_ticks"] = (vwap_session - twap_session) / t

    # Time of day / weekday from the timestamp (UTC)
    if USE_TIME_FEATURES:
        ts = df["timestamp"]
        minute_of_day = ts.dt.hour * 60 + ts.dt.minute
        f["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        f["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
        f["weekday"] = ts.dt.dayofweek
        f["minutes_since_gap"] = minutes_since_gap(ts)

    out = pd.DataFrame(f).replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


def minutes_since_gap(ts: pd.Series) -> pd.Series:
    """Minutes since the last pause > MAX_GAP_MINUTES (session start / after the weekend)."""
    gap = ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES
    block = gap.cumsum()
    return ts.groupby(block).cumcount().astype("float32")


t0 = time.time()
X_train_all = build_features(df_train)
X_test = build_features(df_test)
FEATURE_COLUMNS = list(X_train_all.columns)
print(f"{len(FEATURE_COLUMNS)} inputs built in {time.time() - t0:.0f} s")
print("Indicator inputs:", [c for c in FEATURE_COLUMNS if c.startswith(("rsi_", "dist_twap", "dist_vwap", "vwap_minus"))])

112 inputs built in 5 s
Indicator inputs: ['rsi_3', 'rsi_3_change_5', 'rsi_5', 'rsi_5_change_5', 'rsi_7', 'rsi_7_change_5', 'rsi_9', 'rsi_9_change_5', 'rsi_14', 'rsi_14_change_5', 'rsi_21', 'rsi_21_change_5', 'rsi_30', 'rsi_30_change_5', 'rsi_50', 'rsi_50_change_5', 'rsi_100', 'rsi_100_change_5', 'dist_twap_session_ticks', 'dist_twap_session_rel', 'dist_vwap_session_ticks', 'dist_vwap_session_rel', 'dist_twap_5_ticks', 'dist_twap_5_rel', 'dist_twap_15_ticks', 'dist_twap_15_rel', 'dist_twap_30_ticks', 'dist_twap_30_rel', 'dist_twap_60_ticks', 'dist_twap_60_rel', 'dist_twap_120_ticks', 'dist_twap_120_rel', 'dist_twap_240_ticks', 'dist_twap_240_rel', 'dist_vwap_5_ticks', 'dist_vwap_5_rel', 'dist_vwap_15_ticks', 'dist_vwap_15_rel', 'dist_vwap_30_ticks', 'dist_vwap_30_rel', 'dist_vwap_60_ticks', 'dist_vwap_60_rel', 'dist_vwap_120_ticks', 'dist_vwap_120_rel', 'dist_vwap_240_ticks', 'dist_vwap_240_rel', 'vwap_minus_twap_session_ticks']


## 4. Labels: Barrier Outcomes for Every Candle

For every candle, a long and a short trade entered at its close are played through minute by minute:
target (+`TARGET_FILL_EXTRA_TICKS` beyond it) → win, stop → loss (stop wins if both are touched in the same candle),
neither → closed at the close after `horizon` minutes. Windows that contain a pause are invalid (not traded, not trained).

With `mode: "vol"` target and stop are set per candle from the volatility of the **past** `VOL_LOOKBACK` candles.

**Direction label (for the pure direction test):** is the upper barrier (+target) touched **before** the lower one
(−target)? Only windows where exactly one side is reached first count; time-outs and candles touching both are excluded.
This removes the volatility question – what remains is only "up or down".

In [5]:
EXIT_TARGET, EXIT_STOP, EXIT_TIME = 1, 2, 3


def barrier_ticks(df: pd.DataFrame, setup: dict):
    """Target and stop in ticks per candle (arrays)."""
    n = len(df)
    if setup.get("mode", "fixed") == "fixed":
        return np.full(n, float(setup["target_ticks"])), np.full(n, float(setup["stop_ticks"]))
    ret1 = df["Close"].diff() / TICK_SIZE
    expected_move = (ret1.rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).std() * np.sqrt(setup["horizon"])).to_numpy()
    target = np.maximum(np.round(setup["target_sigma"] * expected_move), MIN_BARRIER_TICKS)
    stop = np.maximum(np.round(setup["stop_sigma"] * expected_move), MIN_BARRIER_TICKS)
    return target, stop


def barrier_outcomes(df: pd.DataFrame, setup: dict) -> dict:
    horizon = setup["horizon"]
    target_ticks, stop_ticks = barrier_ticks(df, setup)
    c = df["Close"].to_numpy("float64")
    h = df["High"].to_numpy("float64")
    l = df["Low"].to_numpy("float64")
    n = len(df)
    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    out = {"target_ticks": target_ticks, "stop_ticks": stop_ticks}
    for side, d in (("long", 1), ("short", -1)):
        target = c + d * (target_ticks * TICK_SIZE + extra)
        stop = c - d * stop_ticks * TICK_SIZE
        result = np.full(n, np.nan)
        exit_type = np.zeros(n, dtype="int8")
        minutes = np.zeros(n, dtype="int16")
        done = np.zeros(n, dtype=bool)
        for k in range(1, horizon + 1):
            hk = np.full(n, np.nan)
            lk = np.full(n, np.nan)
            hk[:n - k], lk[:n - k] = h[k:], l[k:]
            if d == 1:
                stop_hit, tgt_hit = lk <= stop, hk >= target
            else:
                stop_hit, tgt_hit = hk >= stop, lk <= target
            stop_hit &= ~done
            tgt_hit &= ~done & ~stop_hit
            result[stop_hit], exit_type[stop_hit], minutes[stop_hit] = -stop_ticks[stop_hit], EXIT_STOP, k
            result[tgt_hit], exit_type[tgt_hit], minutes[tgt_hit] = target_ticks[tgt_hit], EXIT_TARGET, k
            done |= stop_hit | tgt_hit
        ck = np.full(n, np.nan)
        ck[:n - horizon] = c[horizon:]
        open_ = ~done
        result[open_] = d * (ck[open_] - c[open_]) / TICK_SIZE
        exit_type[open_], minutes[open_] = EXIT_TIME, horizon
        out[side] = {"result": result, "exit": exit_type, "minutes": minutes, "win": exit_type == EXIT_TARGET}

    # Direction label: +target before -target (symmetric, no extra tick); both in one candle / time-out -> unresolved
    up_level, down_level = c + target_ticks * TICK_SIZE, c - target_ticks * TICK_SIZE
    up_first = np.zeros(n, dtype=bool)
    resolved = np.zeros(n, dtype=bool)
    decided = np.zeros(n, dtype=bool)
    for k in range(1, horizon + 1):
        hk = np.full(n, np.nan)
        lk = np.full(n, np.nan)
        hk[:n - k], lk[:n - k] = h[k:], l[k:]
        up, down = (hk >= up_level) & ~decided, (lk <= down_level) & ~decided
        only_up, only_down = up & ~down, down & ~up
        up_first |= only_up
        resolved |= only_up | only_down
        decided |= up | down
    out["direction"] = {"up_first": up_first, "resolved": resolved}

    # Valid = complete window inside the file, no pause inside the window, warm-up done
    gap = (df["timestamp"].diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.cumsum(gap)
    idx = np.arange(n)
    end = np.minimum(idx + horizon, n - 1)
    out["valid"] = ((idx + horizon < n) & ((cum[end] - cum[idx]) == 0) & (idx >= WARMUP_CANDLES)
                    & np.isfinite(target_ticks) & np.isfinite(stop_ticks))
    return out


t0 = time.time()
outcomes_train = {s["name"]: barrier_outcomes(df_train, s) for s in SETUPS}
outcomes_test = {s["name"]: barrier_outcomes(df_test, s) for s in SETUPS}
print(f"Barrier outcomes computed in {time.time() - t0:.0f} s")
for s in SETUPS:
    o = outcomes_train[s["name"]]
    v = o["valid"]
    print(f"[{s['name']}] valid candles {v.sum():,} | win rate if ALWAYS traded: "
          f"long {100 * o['long']['win'][v].mean():.1f} %, short {100 * o['short']['win'][v].mean():.1f} % "
          f"| Ø result long {np.nanmean(o['long']['result'][v]):+.2f}, short {np.nanmean(o['short']['result'][v]):+.2f} ticks (gross)")
    for label, oo in (("training data", o), ("test data", outcomes_test[s["name"]])):
        vv = oo["valid"]
        print(f"      {label}: Ø target {np.mean(oo['target_ticks'][vv]):.1f} / Ø stop {np.mean(oo['stop_ticks'][vv]):.1f} ticks, "
              f"direction resolved in {100 * oo['direction']['resolved'][vv].mean():.1f} % of the windows, "
              f"up first {100 * oo['direction']['up_first'][vv & oo['direction']['resolved']].mean():.1f} %")

Barrier outcomes computed in 4 s
in] valid candles 2,414,279 | win rate if ALWAYS traded: long 24.6 %, short 25.1 % | Ø result long -0.25, short -0.29 ticks (gross)
      training data: Ø target 18.0 / Ø stop 18.0 ticks, direction resolved in 56.2 % of the windows, up first 49.6 %
      test data: Ø target 71.6 / Ø stop 71.6 ticks, direction resolved in 55.5 % of the windows, up first 47.8 %
in] valid candles 2,333,548 | win rate if ALWAYS traded: long 26.6 %, short 27.4 % | Ø result long -0.21, short -0.37 ticks (gross)
      training data: Ø target 36.2 / Ø stop 36.2 ticks, direction resolved in 57.2 % of the windows, up first 49.3 %
      test data: Ø target 144.3 / Ø stop 144.3 ticks, direction resolved in 56.8 % of the windows, up first 46.8 %


## 5. Time Split: Training / Validation

In [6]:
n_train_total = len(df_train)
val_start = int(n_train_total * (1 - VALIDATION_FRACTION))
max_horizon = max(s["horizon"] for s in SETUPS)
# Purge: training labels must not look into the validation period
train_rows = np.arange(0, val_start - max_horizon, TRAIN_ROW_STRIDE)
val_rows = np.arange(val_start, n_train_total)
print(f"Training:   {df_train['timestamp'].iloc[train_rows[0]]} → {df_train['timestamp'].iloc[train_rows[-1]]} "
      f"({len(train_rows):,} candles, stride {TRAIN_ROW_STRIDE})")
print(f"Validation: {df_train['timestamp'].iloc[val_rows[0]]} → {df_train['timestamp'].iloc[val_rows[-1]]} ({len(val_rows):,} candles)")
print(f"Test:       {df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]} ({len(df_test):,} candles)")

Training:   2018-11-25 23:00:00+00:00 → 2024-06-04 04:27:00+00:00 (1,953,097 candles, stride 1)
Validation: 2024-06-04 05:29:00+00:00 → 2025-10-19 23:59:00+00:00 (488,290 candles)
Test:       2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00 (354,649 candles)


## 6. Train Long and Short Classifier per Setup

In [7]:
models = {}
auc_rows = []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    tr = train_rows[o_tr["valid"][train_rows]]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    models[name] = {}
    for side in ("long", "short"):
        t0 = time.time()
        clf = HistGradientBoostingClassifier(**GBM_PARAMS)
        clf.fit(X_train_all.iloc[tr], o_tr[side]["win"][tr])
        p_val = clf.predict_proba(X_train_all.iloc[va])[:, 1]
        p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
        models[name][side] = clf
        auc_rows.append({
            "Setup": name, "Side": side, "Train candles": len(tr),
            "Base win rate val %": 100 * o_tr[side]["win"][va].mean(),
            "AUC validation": roc_auc_score(o_tr[side]["win"][va], p_val),
            "AUC test": roc_auc_score(o_te[side]["win"][te], p_te),
            "Ø P val": p_val.mean(), "P val 99%": np.percentile(p_val, 99),
            "Fit seconds": time.time() - t0,
        })
        print(f"[{name} {side}] trained in {time.time() - t0:.0f} s – AUC validation {auc_rows[-1]['AUC validation']:.4f}, "
              f"test {auc_rows[-1]['AUC test']:.4f}")

print("\nAUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful")
display(pd.DataFrame(auc_rows).round(4))

in long] trained in 33 s – AUC validation 0.6037, test 0.5914
in short] trained in 28 s – AUC validation 0.6127, test 0.5847
in long] trained in 28 s – AUC validation 0.6147, test 0.5864
in short] trained in 29 s – AUC validation 0.6218, test 0.5822

AUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful


,Setup,Side,Train candles,Base win rate val %,AUC validation,AUC test,Ø P val,P val 99%,Fit seconds
0,15min,long,1931344,26.7251,0.6037,0.5914,0.2390,0.4246,33.1362
1,15min,short,1931344,26.9117,0.6127,0.5847,0.2517,0.4242,28.3385
2,60min,long,1866858,28.1934,0.6147,0.5864,0.2390,0.4552,28.4743
3,60min,short,1866858,27.9958,0.6218,0.5822,0.2652,0.4758,29.0802


### 6.1 Which Inputs Matter (permutation importance on a validation sample)

In [8]:
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(0)
    for s in SETUPS:
        name = s["name"]
        va = val_rows[outcomes_train[name]["valid"][val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        for side in ("long", "short"):
            imp = permutation_importance(
                models[name][side], X_train_all.iloc[sample], outcomes_train[name][side]["win"][sample],
                scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
            )
            top = (pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(12))
            print(f"[{name} {side}] AUC loss when an input is shuffled (top 12):")
            print("   " + ", ".join(f"{k} {v:.4f}" for k, v in top.items()))

in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0769, volume_rel_60 0.0160, range_mean_15 0.0146, range_mean_60 0.0141, tod_cos 0.0130, minutes_since_gap 0.0109, tod_sin 0.0076, volume 0.0031, range_mean_5 0.0019, rsi_21 0.0009, rsi_100 0.0009, ask 0.0008
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0722, range_mean_15 0.0251, minutes_since_gap 0.0203, range_mean_60 0.0192, volume_rel_60 0.0157, tod_cos 0.0141, range_mean_5 0.0067, tod_sin 0.0057, volume 0.0039, ret_std_15 0.0017, delta_sum_60 0.0014, dist_low_60_ticks 0.0012
in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0867, minutes_since_gap 0.0258, tod_cos 0.0203, volume_rel_60 0.0082, range_mean_60 0.0082, tod_sin 0.0061, range_mean_15 0.0043, volume 0.0022, delta_sum_60 0.0010, weekday 0.0009, dist_low_60_ticks 0.0008, dist_vwap_240_rel 0.0008
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0764, minutes_since_gap 0.0457, range_mean_60 

### 6.2 Pure Direction Test: "Up or Down First?"

One classifier per setup on the direction label only (volatility removed). **AUC 0.50 = no direction information**,
0.52–0.55 = weak, above 0.55 = interesting. The bucket table shows on the test data whether higher model probabilities
really go along with more "up first" outcomes – that is what a trading signal needs.

In [9]:
direction_models, direction_rows = {}, []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    ok_tr = o_tr["valid"] & o_tr["direction"]["resolved"]
    ok_te = o_te["valid"] & o_te["direction"]["resolved"]
    tr = train_rows[ok_tr[train_rows]]
    va = val_rows[ok_tr[val_rows]]
    te = np.flatnonzero(ok_te)
    t0 = time.time()
    clf = HistGradientBoostingClassifier(**GBM_PARAMS)
    clf.fit(X_train_all.iloc[tr], o_tr["direction"]["up_first"][tr])
    direction_models[name] = clf
    p_va = clf.predict_proba(X_train_all.iloc[va])[:, 1]
    p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
    y_va, y_te = o_tr["direction"]["up_first"][va], o_te["direction"]["up_first"][te]
    direction_rows.append({
        "Setup": name, "Train windows": len(tr), "Up first val %": 100 * y_va.mean(), "Up first test %": 100 * y_te.mean(),
        "AUC direction validation": roc_auc_score(y_va, p_va), "AUC direction test": roc_auc_score(y_te, p_te),
        "Fit seconds": time.time() - t0,
    })
    buckets = pd.qcut(p_te, 10, labels=False, duplicates="drop")
    table = (pd.DataFrame({"bucket": buckets, "p": p_te, "up": y_te})
             .groupby("bucket").agg(windows=("up", "size"), mean_p=("p", "mean"), actual_up_pct=("up", "mean")))
    table["actual_up_pct"] *= 100
    print(f"[{name}] test: model probability (10 buckets, low -> high) vs. actual share 'up first'")
    display(table.round(3))

print("AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting")
direction_df = pd.DataFrame(direction_rows)
display(direction_df.round(4))

# Comparison with the previous run (validation only – the test data is not used for this decision)
for _, r in direction_df.iterrows():
    ref = REFERENCE_DIRECTION_AUC.get(r["Setup"])
    if ref is None:
        continue
    gain = r["AUC direction validation"] - ref
    verdict = ("new periods HELP – keep the most important ones" if gain >= MIN_AUC_GAIN
               else "no relevant gain – keep the previous indicator set")
    print(f"[{r['Setup']}] direction AUC validation {r['AUC direction validation']:.4f} vs. reference {ref:.4f} "
          f"-> {gain:+.4f} → {verdict}")

# Which inputs does the DIRECTION model use? (permutation importance on resolved validation windows)
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(1)
    indicator_prefixes = ("rsi_", "dist_twap", "dist_vwap", "vwap_minus")
    for s in SETUPS:
        name = s["name"]
        o_tr = outcomes_train[name]
        ok = o_tr["valid"] & o_tr["direction"]["resolved"]
        va = val_rows[ok[val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        imp = permutation_importance(
            direction_models[name], X_train_all.iloc[sample], o_tr["direction"]["up_first"][sample],
            scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
        )
        importance = pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False)
        print(f"\n[{name} DIRECTION] top 15 inputs (AUC loss when shuffled):")
        print("   " + ", ".join(f"{k} {v:.4f}" for k, v in importance.head(15).items()))
        ind = importance[[c for c in importance.index if c.startswith(indicator_prefixes)]]
        print(f"[{name} DIRECTION] all RSI / TWAP / VWAP inputs, best first:")
        display(ind.round(5).to_frame("AUC loss").T)

in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19442,0.360,44.985
1,19441,0.408,46.150
2,19442,0.432,47.202
3,19441,0.449,46.479
4,19442,0.464,47.850
5,19441,0.477,49.087
6,19441,0.491,49.097
7,19442,0.505,49.851
8,19441,0.524,48.953


in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19242,0.269,44.954
1,19241,0.342,46.437
2,19241,0.378,46.001
3,19242,0.406,46.009
4,19241,0.430,45.855
5,19241,0.452,47.498
6,19242,0.475,47.220
7,19241,0.500,47.669
8,19241,0.534,48.683


AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting


,Setup,Train windows,Up first val %,Up first test %,AUC direction validation,AUC direction test,Fit seconds
0,15min,1080353,49.9036,47.8425,0.5211,0.5151,13.8398
1,60min,1064210,50.1908,46.7773,0.5208,0.5108,13.6538


in] direction AUC validation 0.5211 vs. reference 0.5228 -> -0.0017 → no relevant gain – keep the previous indicator set
in] direction AUC validation 0.5208 vs. reference 0.5152 -> +0.0056 → new periods HELP – keep the most important ones

in DIRECTION] top 15 inputs (AUC loss when shuffled):
   vwap_minus_twap_session_ticks 0.0058, tod_cos 0.0029, dist_vwap_120_rel 0.0028, dist_vwap_session_ticks 0.0025, tod_sin 0.0025, minutes_since_gap 0.0025, session_delta 0.0019, dist_high_60_ticks 0.0018, pos_in_range_60 0.0017, range_mean_60 0.0016, delta_ratio_60 0.0012, dist_twap_120_ticks 0.0011, dist_high_15_ticks 0.0010, range_mean_15 0.0010, delta_sum_60 0.0009
in DIRECTION] all RSI / TWAP / VWAP inputs, best first:


,vwap_minus_twap_session_ticks,dist_vwap_120_rel,dist_vwap_session_ticks,dist_twap_120_ticks,rsi_100,dist_vwap_60_rel,dist_twap_120_rel,dist_vwap_session_rel,dist_twap_session_rel,dist_twap_60_ticks,dist_twap_60_rel,dist_twap_30_ticks,rsi_21,dist_twap_30_rel,dist_vwap_60_ticks,rsi_14,dist_vwap_5_rel,rsi_9,dist_twap_5_rel,dist_vwap_15_ticks,dist_twap_5_ticks,dist_twap_15_rel,rsi_30_change_5,rsi_5_change_5,dist_vwap_5_ticks,rsi_14_change_5,rsi_50_change_5,rsi_7,rsi_5,dist_twap_session_ticks,rsi_7_change_5,rsi_9_change_5,rsi_21_change_5,rsi_100_change_5,dist_twap_15_ticks,rsi_3,dist_vwap_30_rel,dist_vwap_15_rel,rsi_3_change_5,rsi_50,dist_twap_240_rel,rsi_30,dist_vwap_240_rel,dist_vwap_30_ticks,dist_vwap_120_ticks,dist_twap_240_ticks,dist_vwap_240_ticks
AUC loss,0.00581,0.00278,0.00252,0.00108,0.00086,0.00068,0.00058,0.00049,0.00048,0.00036,0.00034,0.00025,0.00023,0.00019,0.00018,0.00015,0.00014,0.00014,0.00013,0.00011,0.00008,0.00006,0.00006,0.00004,0.00003,0.00002,0.00002,0.00002,0.00002,0.00001,0.00001,0.00001,-0.00001,-0.00002,-0.00004,-0.00004,-0.00004,-0.00005,-0.00005,-0.00009,-0.00011,-0.00017,-0.0002,-0.00023,-0.00045,-0.00055,-0.00095



in DIRECTION] top 15 inputs (AUC loss when shuffled):
   vwap_minus_twap_session_ticks 0.0062, dist_high_60_ticks 0.0060, dist_twap_session_ticks 0.0037, weekday 0.0034, ret_std_60 0.0031, session_delta_pct 0.0030, session_delta 0.0026, dist_twap_session_rel 0.0025, minutes_since_gap 0.0025, dist_vwap_240_rel 0.0020, tod_sin 0.0018, rsi_100 0.0016, range_mean_60 0.0013, delta_ratio_60 0.0012, dist_low_60_ticks 0.0011
in DIRECTION] all RSI / TWAP / VWAP inputs, best first:


,vwap_minus_twap_session_ticks,dist_twap_session_ticks,dist_twap_session_rel,dist_vwap_240_rel,rsi_100,dist_twap_240_rel,dist_vwap_session_ticks,dist_vwap_session_rel,dist_vwap_240_ticks,dist_vwap_60_ticks,dist_vwap_30_ticks,dist_vwap_60_rel,dist_twap_30_rel,dist_twap_120_rel,dist_twap_60_rel,dist_vwap_30_rel,rsi_5,dist_vwap_15_ticks,rsi_21_change_5,rsi_14,rsi_50,dist_twap_15_ticks,rsi_50_change_5,rsi_9_change_5,rsi_7_change_5,rsi_14_change_5,dist_twap_5_ticks,rsi_100_change_5,rsi_3,rsi_3_change_5,rsi_7,rsi_5_change_5,rsi_30_change_5,dist_twap_15_rel,rsi_21,rsi_9,dist_vwap_15_rel,rsi_30,dist_vwap_5_rel,dist_vwap_5_ticks,dist_twap_5_rel,dist_twap_30_ticks,dist_vwap_120_ticks,dist_twap_240_ticks,dist_twap_60_ticks,dist_twap_120_ticks,dist_vwap_120_rel
AUC loss,0.00625,0.00369,0.00251,0.00199,0.00159,0.00094,0.00092,0.00055,0.00048,0.00039,0.0003,0.00024,0.00014,0.00011,0.00009,0.00008,0.00002,0.00001,0.00001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-0.00001,-0.00001,-0.00002,-0.00002,-0.00003,-0.00004,-0.00004,-0.00007,-0.0001,-0.00021,-0.00025,-0.00025,-0.00034,-0.00056,-0.00059


## 7. Trading Simulation and Threshold Selection (validation only)

One position at a time. At every candle: `P_long`, `P_short`; if the larger one is ≥ threshold, a trade in that direction is
opened at the close; its result and duration come from the barrier outcome of that candle. Net = gross − costs.
The threshold with the best **t-value** (Ø net ÷ std × √trades) and at least `MIN_VALIDATION_TRADES` trades is selected.

In [10]:
def simulate(outcome: dict, rows: np.ndarray, direction: np.ndarray) -> pd.DataFrame:
    """rows: candidate candles in time order, direction: +1 / -1 per candidate."""
    trades, next_free = [], -1
    for r, d in zip(rows, direction):
        if r < next_free:
            continue
        side = outcome["long"] if d == 1 else outcome["short"]
        trades.append({"t": int(r), "direction": int(d), "gross_ticks": float(side["result"][r]),
                       "exit": int(side["exit"][r]), "minutes": int(side["minutes"][r])})
        next_free = r + int(side["minutes"][r])
    return pd.DataFrame(trades)


def summarize(trades: pd.DataFrame, timestamps: pd.Series, days: int) -> dict:
    """timestamps: all candles of the data set (trade rows index into it), days: trading days of the period."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.concatenate([[0.0], np.cumsum(net * TICK_VALUE_USD)])
    drawdown = (np.maximum.accumulate(equity) - equity).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    ts = timestamps.iloc[trades["t"].to_numpy()]
    monthly = pd.Series(net, index=ts.dt.strftime("%Y-%m").to_numpy()).groupby(level=0).sum()
    days = max(days, 1)
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades), "Trades/day": len(trades) / days, "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == EXIT_TARGET).mean(),
        "Stop %": 100 * (trades["exit"] == EXIT_STOP).mean(),
        "Time exit %": 100 * (trades["exit"] == EXIT_TIME).mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
    }


def signals(p_long: np.ndarray, p_short: np.ndarray, threshold: float):
    best = np.maximum(p_long, p_short)
    mask = best >= threshold
    direction = np.where(p_long >= p_short, 1, -1)
    return mask, direction


probs = {"val": {}, "test": {}}
chosen = {}
for s in SETUPS:
    name = s["name"]
    o_tr = outcomes_train[name]
    va = val_rows[o_tr["valid"][val_rows]]
    p_l = models[name]["long"].predict_proba(X_train_all.iloc[va])[:, 1]
    p_s = models[name]["short"].predict_proba(X_train_all.iloc[va])[:, 1]
    probs["val"][name] = (va, p_l, p_s)

    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(o_tr, va[mask], direction[mask])
        rows.append({"Threshold": thr, "Signal candles": int(mask.sum()),
                     **summarize(tr, df_train["timestamp"], val_days)})
    val_table = pd.DataFrame(rows)
    eligible = val_table[val_table["Trades"] >= MIN_VALIDATION_TRADES]
    if eligible.empty or eligible["t-value"].isna().all():
        chosen[name] = None
        print(f"[{name}] no threshold with ≥ {MIN_VALIDATION_TRADES} validation trades – setup not tradable")
    else:
        best = eligible.loc[eligible["t-value"].idxmax()]
        chosen[name] = float(best["Threshold"])
        print(f"[{name}] chosen threshold {chosen[name]:.3f}: validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
              f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")
    print(f"[{name}] validation results per threshold:")
    display(val_table.round(3))

in] chosen threshold 0.500: validation Ø net +1.28 ticks, t-value 0.48, 135 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,151526,25335,58.919,40.600,36.408,36.621,26.971,-1.983,-1.630,-2.225,0.836,-11.592,0/17,-502490.0,515490.0
1,0.325,101828,19447,45.226,40.983,38.047,37.965,23.988,-1.909,-1.741,-2.025,0.844,-9.714,1/17,-371150.0,386850.0
2,0.350,63274,13712,31.888,42.569,39.462,39.630,20.909,-2.020,-2.046,-2.001,0.839,-8.572,0/17,-277000.0,281980.0
3,0.375,36136,9038,21.019,44.811,40.440,41.746,17.814,-2.316,-2.274,-2.350,0.820,-7.965,1/17,-209340.0,212610.0
4,0.400,18572,5277,12.272,47.925,42.562,42.979,14.459,-1.988,-1.726,-2.229,0.846,-5.171,1/17,-104910.0,106530.0
5,0.425,8212,2814,6.544,52.665,44.741,43.497,11.763,-1.216,-0.637,-1.860,0.903,-2.329,4/17,-34210.0,38710.0
6,0.450,2898,1226,2.851,56.688,46.003,45.351,8.646,-0.831,-0.364,-1.443,0.933,-1.054,7/17,-10190.0,12850.0
7,0.475,831,440,1.023,57.273,47.500,45.455,7.045,-0.780,0.972,-3.128,0.940,-0.573,7/17,-3430.0,8440.0
8,0.500,225,135,0.314,71.111,53.333,42.222,4.444,1.281,2.865,-2.615,1.099,0.483,9/17,1730.0,3740.0
9,0.525,84,52,0.121,84.615,55.769,40.385,3.846,3.692,3.273,6.000,1.316,0.894,9/15,1920.0,1870.0


in] chosen threshold 0.525: validation Ø net +0.10 ticks, t-value 0.02, 187 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,187289,8068,18.763,34.817,36.254,35.783,27.962,-2.041,-0.671,-2.773,0.907,-3.557,2/17,-164660.0,173690.0
1,0.325,146163,6881,16.002,35.678,37.611,37.407,24.982,-2.011,-1.473,-2.310,0.908,-3.295,4/17,-138400.0,161980.0
2,0.350,108163,5686,13.223,36.159,39.377,38.656,21.966,-1.795,-0.963,-2.266,0.917,-2.726,3/17,-102060.0,114670.0
3,0.375,74620,4449,10.347,35.154,41.807,38.278,19.915,-0.838,0.564,-1.598,0.960,-1.141,7/17,-37290.0,44060.0
4,0.400,47946,3213,7.472,35.325,42.359,39.963,17.678,-1.214,-0.915,-1.377,0.942,-1.423,8/17,-38990.0,53060.0
5,0.425,28633,2202,5.121,34.196,43.279,41.054,15.668,-0.881,0.586,-1.643,0.956,-0.881,7/17,-19390.0,25830.0
6,0.450,14908,1436,3.340,34.331,44.847,41.156,13.997,-0.780,0.081,-1.230,0.961,-0.627,7/17,-11200.0,24980.0
7,0.475,6697,803,1.867,36.364,44.085,43.711,12.204,-2.497,0.031,-3.941,0.886,-1.436,7/17,-20050.0,27990.0
8,0.500,2788,416,0.967,32.212,43.029,44.952,12.019,-4.673,-1.828,-6.025,0.816,-1.751,5/17,-19440.0,21220.0
9,0.525,1145,187,0.435,33.155,46.524,42.246,11.230,0.102,-0.145,0.224,1.004,0.025,9/17,190.0,8790.0


## 8. Test: Model vs. Random Direction vs. Always Long

Applied **once** with the threshold chosen on validation. The comparisons use exactly the same entry candles.

In [11]:
test_days = df_test["timestamp"].dt.date.nunique()
test_rows = []
test_trades = {}
for s in SETUPS:
    name = s["name"]
    o_te = outcomes_test[name]
    te = np.flatnonzero(o_te["valid"])
    p_l = models[name]["long"].predict_proba(X_test.iloc[te])[:, 1]
    p_s = models[name]["short"].predict_proba(X_test.iloc[te])[:, 1]
    probs["test"][name] = (te, p_l, p_s)
    thr = chosen[name]
    if thr is None:
        continue
    mask, direction = signals(p_l, p_s, thr)
    rows_sig = te[mask]
    strategies = {
        "Model": [direction[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=mask.sum()) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(mask.sum(), dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for d in dirs:
            tr = simulate(o_te, rows_sig, d)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Model":
                test_trades[name] = tr
        row = {"Setup": name, "Threshold": thr, "Strategy": strategy, "Signal candles": int(mask.sum())}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        test_rows.append(row)

test_df = pd.DataFrame(test_rows)
print(f"Rules: costs {COST_PER_TRADE_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, stop first in the same "
      f"candle, no trades across pauses > {MAX_GAP_MINUTES} min, tick value {TICK_VALUE_USD} USD")
display(test_df.round(2))

Rules: costs 2.0 ticks, target fill +1 tick, stop first in the same candle, no trades across pauses > 5 min, tick value 10.0 USD


,Setup,Threshold,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,0.50,Model,212,106.0,0.34,74.53,43.40,45.28,11.32,-0.18,-0.39,0.44,0.99,-0.04,7/13,-190.0,5540.0
1,15min,0.50,Random direction (Ø 5),212,106.8,0.34,52.62,46.63,42.70,10.67,-1.38,0.22,-3.15,0.95,-0.30,4/13,-1474.0,5808.0
2,15min,0.50,Always long,212,107.0,0.34,100.00,44.86,43.93,11.21,-0.93,-0.93,NaN,0.96,-0.20,5/13,-1000.0,5500.0
3,60min,0.52,Model,1620,231.0,0.74,11.26,35.50,44.16,20.35,-14.44,-26.19,-12.95,0.75,-1.94,5/13,-33350.0,42240.0
4,60min,0.52,Random direction (Ø 5),1620,231.0,0.74,51.95,41.21,38.44,20.35,1.81,8.07,-4.91,1.04,0.24,4/13,4180.0,18036.0
5,60min,0.52,Always long,1620,231.0,0.74,100.00,41.99,37.66,20.35,4.61,4.61,NaN,1.10,0.62,7/13,10640.0,13140.0


### 8.1 For Information Only: All Thresholds on the Test Data

Do **not** pick the threshold from this table – that would be fitting to the test year. It only shows whether the
result is stable around the chosen threshold.

In [12]:
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(outcomes_test[name], te[mask], direction[mask])
        rows.append({"Threshold": thr, **summarize(tr, df_test["timestamp"], test_days)})
    print(f"[{name}] test results per threshold (chosen on validation: {chosen[name]})")
    display(pd.DataFrame(rows).round(3))

in] test results per threshold (chosen on validation: 0.5)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,17804,56.882,35.801,35.441,34.964,29.594,-1.807,-0.869,-2.329,0.931,-3.670,1/13,-321630.0,332990.0
1,0.325,13322,42.562,37.217,37.224,36.346,26.430,-1.595,-1.656,-1.558,0.937,-2.917,2/13,-212440.0,222520.0
2,0.350,9199,29.390,39.896,38.950,37.341,23.709,-1.421,-0.508,-2.027,0.943,-2.243,5/13,-130710.0,156110.0
3,0.375,5775,18.450,44.104,40.745,38.061,21.195,-0.860,0.052,-1.580,0.964,-1.142,6/13,-49660.0,76470.0
4,0.400,3317,10.597,47.633,41.302,39.343,19.355,-0.686,-0.930,-0.463,0.971,-0.709,5/13,-22740.0,53860.0
5,0.425,1734,5.540,52.595,43.137,40.023,16.840,0.339,-0.333,1.084,1.015,0.262,7/13,5870.0,21340.0
6,0.450,829,2.649,56.092,44.753,40.048,15.199,1.037,-0.191,2.607,1.048,0.574,7/13,8600.0,20750.0
7,0.475,311,0.994,63.987,42.122,44.051,13.826,-1.826,-2.955,0.179,0.918,-0.681,5/13,-5680.0,12490.0
8,0.500,106,0.339,74.528,43.396,45.283,11.321,-0.179,-0.392,0.444,0.992,-0.038,7/13,-190.0,5540.0
9,0.525,44,0.141,75.000,52.273,38.636,9.091,3.409,-0.576,15.364,1.190,0.535,6/11,1500.0,2040.0


in] test results per threshold (chosen on validation: 0.525)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,5612,17.930,22.915,34.711,34.533,30.756,-1.465,-3.245,-0.936,0.969,-0.936,4/13,-82240.0,121370.0
1,0.325,4790,15.304,23.173,35.866,35.240,28.894,-1.109,-3.239,-0.467,0.976,-0.670,5/13,-53130.0,100460.0
2,0.350,3891,12.431,22.719,36.880,36.520,26.600,-1.531,-0.410,-1.861,0.968,-0.849,4/13,-59570.0,89720.0
3,0.375,3029,9.677,22.912,37.669,37.042,25.289,-1.859,-4.182,-1.169,0.960,-0.938,5/13,-56320.0,91810.0
4,0.400,2210,7.061,21.719,38.824,38.326,22.851,-1.538,-0.271,-1.890,0.967,-0.671,6/13,-34000.0,71800.0
5,0.425,1497,4.783,21.844,39.947,38.811,21.242,-1.140,-0.346,-1.362,0.975,-0.405,6/13,-17070.0,51530.0
6,0.450,962,3.073,20.998,39.709,40.333,19.958,-3.843,-3.535,-3.925,0.922,-1.040,4/13,-36970.0,72680.0
7,0.475,592,1.891,17.061,40.878,40.709,18.412,-3.529,3.158,-4.904,0.930,-0.733,6/13,-20890.0,47840.0
8,0.500,361,1.153,13.019,37.673,43.213,19.114,-7.598,-6.149,-7.815,0.858,-1.267,7/13,-27430.0,45090.0
9,0.525,231,0.738,11.255,35.498,44.156,20.346,-14.437,-26.192,-12.946,0.749,-1.942,5/13,-33350.0,42240.0


## 8.2 Direction Strategy: Direction Model + Timing

* **Direction:** short if `P(up first)` is in the lowest `X %`, long if it is in the highest `X %` (cut-offs from the
  validation distribution, `DIRECTION_QUANTILES`).
* **Timing:** additionally the win model of the traded side must be at least at its validation quantile
  `TIMING_QUANTILES` (0 = no filter) – i.e. a move large enough to reach the target is expected.
* **Sides:** both, short only, long only (`DIRECTION_SIDES`).
* Same exits as above (volatility target / stop, stop first in the same candle, +1 tick for the target, time exit),
  one position at a time, costs included.

All combinations are evaluated on the **validation period**; the best t-value (with at least `MIN_VALIDATION_TRADES`
trades) is applied **once** to the test data and compared with random direction, always long and always short on exactly
the same entry candles. Long and short are reported separately.

In [13]:
def direction_signals(p_up, p_long, p_short, cfg):
    long_ = (p_up >= cfg["p_up_high"]) & (p_long >= cfg["p_long_min"])
    short_ = (p_up <= cfg["p_up_low"]) & (p_short >= cfg["p_short_min"])
    if cfg["side"] == "short":
        long_[:] = False
    elif cfg["side"] == "long":
        short_[:] = False
    return long_ | short_, np.where(long_, 1, -1)


dir_chosen, dir_test_rows, dir_trades, dir_signal_rows = {}, [], {}, {}
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    pu_va = direction_models[name].predict_proba(X_train_all.iloc[va])[:, 1]
    pu_te = direction_models[name].predict_proba(X_test.iloc[te])[:, 1]
    pl_va, ps_va = probs["val"][name][1], probs["val"][name][2]
    pl_te, ps_te = probs["test"][name][1], probs["test"][name][2]
    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()

    # ---- grid on validation
    grid = []
    for q in DIRECTION_QUANTILES:
        for tq in TIMING_QUANTILES:
            for side in DIRECTION_SIDES:
                cfg = {
                    "direction_quantile": q, "timing_quantile": tq, "side": side,
                    "p_up_low": float(np.quantile(pu_va, q)), "p_up_high": float(np.quantile(pu_va, 1 - q)),
                    "p_long_min": float(np.quantile(pl_va, tq)) if tq > 0 else -np.inf,
                    "p_short_min": float(np.quantile(ps_va, tq)) if tq > 0 else -np.inf,
                }
                mask, d = direction_signals(pu_va, pl_va, ps_va, cfg)
                tr = simulate(o_tr, va[mask], d[mask])
                grid.append({**cfg, **summarize(tr, df_train["timestamp"], val_days)})
    grid_df = pd.DataFrame(grid)
    eligible = grid_df[(grid_df["Trades"] >= MIN_VALIDATION_TRADES) & grid_df["t-value"].notna()]
    print(f"[{name}] validation: all combinations")
    display(grid_df.drop(columns=["p_long_min", "p_short_min"]).round(3))
    if eligible.empty:
        print(f"[{name}] no combination with ≥ {MIN_VALIDATION_TRADES} validation trades")
        continue
    best = eligible.loc[eligible["t-value"].idxmax()]
    cfg = {k: best[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
    dir_chosen[name] = cfg
    print(f"[{name}] chosen on validation: side {cfg['side']}, direction {cfg['direction_quantile']:.0%}, "
          f"timing quantile {cfg['timing_quantile']} -> validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
          f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")

    # ---- test, applied once
    mask, d = direction_signals(pu_te, pl_te, ps_te, cfg)
    rows_sig = te[mask]
    dir_signal_rows[name] = (rows_sig, d[mask])
    n_sig = int(mask.sum())
    strategies = {
        "Direction model": [d[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=n_sig) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(n_sig, dtype=int)],
        "Always short": [-np.ones(n_sig, dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for dd in dirs:
            tr = simulate(o_te, rows_sig, dd)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Direction model":
                dir_trades[name] = tr
        row = {"Setup": name, "Side": cfg["side"], "Direction quantile": cfg["direction_quantile"],
               "Timing quantile": cfg["timing_quantile"], "Strategy": strategy, "Signal candles": n_sig}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        dir_test_rows.append(row)

    # ---- for information only: all combinations on the test data
    info = []
    for _, g in grid_df.iterrows():
        c = {k: g[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
        m2, d2 = direction_signals(pu_te, pl_te, ps_te, c)
        info.append({"direction_quantile": c["direction_quantile"], "timing_quantile": c["timing_quantile"],
                     "side": c["side"], **summarize(simulate(o_te, te[m2], d2[m2]), df_test["timestamp"], test_days)})
    dir_signal_rows[name + "_info"] = pd.DataFrame(info)

print("TEST – direction strategy with the combination chosen on validation:")
dir_test_df = pd.DataFrame(dir_test_rows)
display(dir_test_df.round(2))
for s in SETUPS:
    if s["name"] + "_info" in dir_signal_rows:
        print(f"[{s['name']}] for information only – all combinations on the test data (do not pick from here):")
        display(dir_signal_rows[s["name"] + "_info"].round(3))

in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.416,0.553,9920,23.070,55.796,29.405,27.187,43.407,-0.735,-0.382,-1.179,0.944,-2.109,4/17,-72870.0,87010.0
1,0.05,0.00,short,0.416,0.553,4462,10.377,0.000,28.194,26.356,45.450,-1.175,NaN,-1.175,0.931,-1.797,4/17,-52440.0,64290.0
2,0.05,0.00,long,0.416,0.553,5579,12.974,100.000,30.203,27.711,42.086,-0.384,-0.384,NaN,0.961,-1.152,6/17,-21450.0,27780.0
3,0.05,0.50,both,0.416,0.553,7047,16.388,59.543,33.489,30.751,35.760,-0.900,-0.324,-1.747,0.934,-2.182,4/17,-63400.0,70220.0
4,0.05,0.50,short,0.416,0.553,2868,6.670,0.000,32.706,31.206,36.088,-1.721,NaN,-1.721,0.908,-2.042,4/17,-49370.0,54320.0
5,0.05,0.50,long,0.416,0.553,4202,9.772,100.000,34.079,30.319,35.602,-0.321,-0.321,NaN,0.969,-0.835,7/17,-13490.0,23550.0
6,0.05,0.75,both,0.416,0.553,4807,11.179,62.388,36.384,33.618,29.998,-0.867,-0.594,-1.319,0.938,-1.765,3/17,-41660.0,46620.0
7,0.05,0.75,short,0.416,0.553,1816,4.223,0.000,36.289,33.645,30.066,-1.288,NaN,-1.288,0.932,-1.230,3/17,-23390.0,28300.0
8,0.05,0.75,long,0.416,0.553,3001,6.979,100.000,36.388,33.489,30.123,-0.585,-0.585,NaN,0.946,-1.255,6/17,-17560.0,25510.0
9,0.10,0.00,both,0.439,0.537,18632,43.330,54.261,29.825,28.038,42.137,-1.194,-0.937,-1.499,0.904,-5.190,2/17,-222520.0,238700.0


in] chosen on validation: side long, direction 5%, timing quantile 0.5 -> validation Ø net -0.32 ticks, t-value -0.83, 4202 trades
in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.345,0.582,3236,7.526,56.613,28.090,26.205,45.705,-1.329,1.043,-4.425,0.949,-1.136,5/17,-43020.0,68410.0
1,0.05,0.00,short,0.345,0.582,1445,3.360,0.000,26.298,26.159,47.543,-4.518,NaN,-4.518,0.872,-2.057,6/17,-65290.0,81980.0
2,0.05,0.00,long,0.345,0.582,1869,4.347,100.000,29.106,25.789,45.104,1.341,1.341,NaN,1.072,1.181,11/17,25070.0,16590.0
3,0.05,0.50,both,0.345,0.582,2370,5.512,58.945,30.802,28.608,40.591,-2.154,-0.017,-5.221,0.916,-1.657,7/17,-51040.0,68700.0
4,0.05,0.50,short,0.345,0.582,987,2.295,0.000,28.875,29.078,42.047,-4.965,NaN,-4.965,0.859,-1.917,6/17,-49000.0,64150.0
5,0.05,0.50,long,0.345,0.582,1407,3.272,100.000,32.054,28.358,39.588,0.095,0.095,NaN,1.005,0.077,10/17,1340.0,16090.0
6,0.05,0.75,both,0.345,0.582,1450,3.372,62.000,34.138,31.793,34.069,-2.174,-0.264,-5.290,0.915,-1.348,9/17,-31520.0,40540.0
7,0.05,0.75,short,0.345,0.582,556,1.293,0.000,31.115,31.655,37.230,-5.396,NaN,-5.396,0.848,-1.596,8/17,-30000.0,40560.0
8,0.05,0.75,long,0.345,0.582,903,2.100,100.000,35.770,32.115,32.115,-0.410,-0.410,NaN,0.979,-0.264,8/17,-3700.0,16900.0
9,0.10,0.00,both,0.384,0.553,5784,13.451,55.861,30.152,27.351,42.497,-1.057,1.107,-3.795,0.955,-1.339,5/17,-61130.0,81750.0


in] chosen on validation: side long, direction 10%, timing quantile 0.0 -> validation Ø net +1.31 ticks, t-value 1.61, 3361 trades
TEST – direction strategy with the combination chosen on validation:


,Setup,Side,Direction quantile,Timing quantile,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,long,0.05,0.5,Direction model,11824,3028.0,9.67,100.00,32.50,32.33,35.17,-1.17,-1.17,NaN,0.94,-1.19,5/13,-35280.0,79100.0
1,15min,long,0.05,0.5,Random direction (Ø 5),11824,3025.0,9.66,49.89,32.40,32.55,35.05,-1.95,-0.83,-3.06,0.91,-1.98,3/13,-59008.0,66744.0
2,15min,long,0.05,0.5,Always long,11824,3028.0,9.67,100.00,32.50,32.33,35.17,-1.17,-1.17,NaN,0.94,-1.19,5/13,-35280.0,79100.0
3,15min,long,0.05,0.5,Always short,11824,3021.0,9.65,0.00,31.35,33.57,35.09,-3.40,NaN,-3.40,0.84,-3.45,3/13,-102720.0,103300.0
4,60min,long,0.10,0.0,Direction model,36914,2397.0,7.66,100.00,29.16,30.16,40.68,-1.77,-1.77,NaN,0.96,-0.70,5/13,-42450.0,75780.0
5,60min,long,0.10,0.0,Random direction (Ø 5),36914,2392.0,7.64,49.71,28.78,30.40,40.82,-2.76,-2.27,-3.23,0.94,-1.09,7/13,-65932.0,97866.0
6,60min,long,0.10,0.0,Always long,36914,2397.0,7.66,100.00,29.16,30.16,40.68,-1.77,-1.77,NaN,0.96,-0.70,5/13,-42450.0,75780.0
7,60min,long,0.10,0.0,Always short,36914,2388.0,7.63,0.00,29.65,29.40,40.95,-2.81,NaN,-2.81,0.94,-1.11,4/13,-67060.0,84480.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,14692,46.939,27.852,28.995,27.001,44.004,-0.648,-1.626,-0.271,0.978,-0.979,6/13,-95240.0,200370.0
1,0.05,0.00,short,10660,34.058,0.000,28.977,25.741,45.281,-0.232,NaN,-0.232,0.993,-0.272,5/13,-24690.0,107480.0
2,0.05,0.00,long,4134,13.208,100.000,29.052,30.019,40.929,-1.383,-1.383,NaN,0.934,-1.604,5/13,-57190.0,105900.0
3,0.05,0.50,both,10026,32.032,30.162,32.476,30.411,37.114,-1.039,-1.214,-0.963,0.965,-1.335,3/13,-104170.0,154620.0
4,0.05,0.50,short,7029,22.457,0.000,32.537,29.549,37.914,-0.863,NaN,-0.863,0.974,-0.840,6/13,-60670.0,98060.0
5,0.05,0.50,long,3028,9.674,100.000,32.497,32.332,35.172,-1.165,-1.165,NaN,0.944,-1.185,5/13,-35280.0,79100.0
6,0.05,0.75,both,6147,19.639,34.895,35.351,32.894,31.755,-0.351,-0.884,-0.066,0.987,-0.386,7/13,-21600.0,95560.0
7,0.05,0.75,short,4014,12.824,0.000,35.501,32.287,32.212,-0.041,NaN,-0.041,0.999,-0.033,7/13,-1630.0,60080.0
8,0.05,0.75,long,2150,6.869,100.000,35.070,34.000,30.930,-0.957,-0.957,NaN,0.954,-0.823,6/13,-20580.0,56010.0
9,0.10,0.00,both,21485,68.642,30.175,29.504,27.903,42.593,-1.166,-1.955,-0.825,0.958,-2.306,2/13,-250570.0,311790.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,4569,14.597,33.815,29.875,27.665,42.460,0.228,-0.028,0.360,1.004,0.103,7/13,10440.0,126150.0
1,0.05,0.00,short,3126,9.987,0.000,29.878,27.031,43.090,-0.176,NaN,-0.176,0.997,-0.061,6/13,-5490.0,119920.0
2,0.05,0.00,long,1640,5.240,100.000,28.841,29.085,42.073,-1.004,-1.004,NaN,0.979,-0.310,7/13,-16470.0,47750.0
3,0.05,0.50,both,3218,10.281,33.810,31.200,31.666,37.135,-3.241,-1.585,-4.086,0.941,-1.322,3/13,-104290.0,130120.0
4,0.05,0.50,short,2156,6.888,0.000,31.494,31.679,36.827,-4.118,NaN,-4.118,0.931,-1.288,2/13,-88780.0,121600.0
5,0.05,0.50,long,1112,3.553,100.000,30.845,32.194,36.960,-1.792,-1.792,NaN,0.960,-0.503,5/13,-19930.0,42410.0
6,0.05,0.75,both,2021,6.457,33.053,33.795,34.290,31.915,-3.664,-5.558,-2.728,0.931,-1.277,4/13,-74040.0,104980.0
7,0.05,0.75,short,1370,4.377,0.000,34.161,33.577,32.263,-3.142,NaN,-3.142,0.944,-0.874,5/13,-43050.0,90950.0
8,0.05,0.75,long,676,2.160,100.000,32.988,36.391,30.621,-6.118,-6.118,NaN,0.871,-1.327,5/13,-41360.0,46110.0
9,0.10,0.00,both,6391,20.419,33.046,30.762,29.213,40.025,-0.255,-1.505,0.362,0.995,-0.140,6/13,-16310.0,175900.0


## 9. Write Test CSV with Signals, Save Models and Log

In [14]:
# Output = original test CSV (all columns as text, timestamp unchanged) + probabilities and signals per setup
test_text = pd.read_csv(TEST_CSV_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
add = pd.DataFrame({"_bar": df_test["Bar"].to_numpy("int64")})
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    col_l, col_s, col_sig = f"P_long_{name}", f"P_short_{name}", f"Signal_{name}"
    add[col_l] = np.nan
    add[col_s] = np.nan
    add.loc[te, col_l] = np.round(p_l, 4)
    add.loc[te, col_s] = np.round(p_s, 4)
    add[col_sig] = 0
    if chosen[name] is not None:
        mask, direction = signals(p_l, p_s, chosen[name])
        add.loc[te[mask], col_sig] = direction[mask]
    add[f"Target_ticks_{name}"] = outcomes_test[name]["target_ticks"]
    add[f"Stop_ticks_{name}"] = outcomes_test[name]["stop_ticks"]
    add[f"P_up_first_{name}"] = np.nan
    add.loc[te, f"P_up_first_{name}"] = np.round(direction_models[name].predict_proba(X_test.iloc[te])[:, 1], 4)
    add[f"Trade_{name}"] = 0                         # trades actually taken (one position at a time)
    if name in test_trades and not test_trades[name].empty:
        add.loc[test_trades[name]["t"].to_numpy(), f"Trade_{name}"] = test_trades[name]["direction"].to_numpy()
    add[f"DirSignal_{name}"] = 0                     # direction strategy (section 8.2): signal / trades taken
    add[f"DirTrade_{name}"] = 0
    if name in dir_signal_rows:
        rows_sig, dirs = dir_signal_rows[name]
        add.loc[rows_sig, f"DirSignal_{name}"] = dirs
    if name in dir_trades and not dir_trades[name].empty:
        add.loc[dir_trades[name]["t"].to_numpy(), f"DirTrade_{name}"] = dir_trades[name]["direction"].to_numpy()

test_text["_bar"] = pd.to_numeric(test_text["Bar"]).astype("int64")
out = test_text.merge(add, on="_bar", how="left", validate="one_to_one").drop(columns="_bar")
out.to_csv(TEST_OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"✓ Written: {TEST_OUTPUT_CSV_PATH} ({len(out):,} rows, new columns: {list(add.columns[1:])})")

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_rows = []
for s in SETUPS:
    name = s["name"]
    path = MODEL_DIR / f"barrier_{SYMBOL}_{name}_{run_ts}.joblib"
    joblib.dump({
        "models": models[name], "direction_model": direction_models.get(name),
        "direction_strategy": dir_chosen.get(name),
        "features": FEATURE_COLUMNS, "setup": s, "threshold": chosen[name],
        "vol_lookback": VOL_LOOKBACK, "min_barrier_ticks": MIN_BARRIER_TICKS,
        "trained_until": str(df_train["timestamp"].iloc[val_start - 1]), "input_columns": INPUT_COLUMNS,
        "use_time_features": USE_TIME_FEATURES, "tick_size": TICK_SIZE,
        "rsi_periods": RSI_PERIODS, "twap_windows": TWAP_WINDOWS, "vwap_windows": VWAP_WINDOWS,
    }, path)
    joblib.dump(joblib.load(path), MODEL_DIR / f"barrier_{SYMBOL}_{name}_latest.joblib")
    print(f"✓ Saved: {path.resolve()}")
    model_row = test_df[(test_df["Setup"] == name) & (test_df["Strategy"] == "Model")]
    auc = pd.DataFrame(auc_rows)
    log_rows.append({
        "run_timestamp": run_ts, "setup": name, "horizon": s["horizon"], "mode": s.get("mode", "fixed"),
        "target": s.get("target_ticks", s.get("target_sigma")), "stop": s.get("stop_ticks", s.get("stop_sigma")),
        "threshold": chosen[name],
        "auc_direction_val": next((r["AUC direction validation"] for r in direction_rows if r["Setup"] == name), None),
        "auc_direction_test": next((r["AUC direction test"] for r in direction_rows if r["Setup"] == name), None),
        "auc_val_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC validation"].iloc[0]),
        "auc_val_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC validation"].iloc[0]),
        "auc_test_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC test"].iloc[0]),
        "auc_test_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC test"].iloc[0]),
        **({k: model_row.iloc[0][k] for k in ["Trades", "Ø net (ticks)", "Profit factor", "t-value",
                                             "Months in profit", "Net total USD/contract", "Max drawdown USD"]}
           if len(model_row) else {}),
        **({f"dir_{k}": dir_chosen[name][k] for k in ["side", "direction_quantile", "timing_quantile"]}
           if name in dir_chosen else {}),
        **({f"dir_{k}": v for k, v in dir_test_df[(dir_test_df["Setup"] == name)
                                                    & (dir_test_df["Strategy"] == "Direction model")].iloc[0][
               ["Trades", "Ø net (ticks)", "Profit factor", "t-value", "Months in profit", "Net total USD/contract"]].items()}
           if len(dir_test_df) and ((dir_test_df["Setup"] == name) & (dir_test_df["Strategy"] == "Direction model")).any() else {}),
        "costs_ticks": COST_PER_TRADE_TICKS, "train_row_stride": TRAIN_ROW_STRIDE, "model_path": str(path.resolve()),
    })
log_df = pd.DataFrame(log_rows)
if LOG_PATH.exists():
    log_df = pd.concat([pd.read_csv(LOG_PATH, sep=";"), log_df], ignore_index=True)
log_df.to_csv(LOG_PATH, sep=";", index=False)
print(f"✓ Logged to {LOG_PATH.resolve()}")

✓ Written: E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026_barrier_indicators_signals.csv (354,649 rows, new columns: ['P_long_15min', 'P_short_15min', 'Signal_15min', 'Target_ticks_15min', 'Stop_ticks_15min', 'P_up_first_15min', 'Trade_15min', 'DirSignal_15min', 'DirTrade_15min', 'P_long_60min', 'P_short_60min', 'Signal_60min', 'Target_ticks_60min', 'Stop_ticks_60min', 'P_up_first_60min', 'Trade_60min', 'DirSignal_60min', 'DirTrade_60min'])
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_indicators\barrier_GC_15min_20261026_201431.joblib
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_indicators\barrier_GC_60min_20261026_201431.joblib
✓ Logged to E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier_indicators\barrier_log.csv
